# Modelling Inflation with Random Forests

created by: Mateusz Dzięgielewski

# Setting up the workspace

## Importing Libraries

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer

from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor

import pmdarima as pm

## Setting plot style

In [7]:
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
save_dpi = 150
rf_plot_color = "#009E73"

## Loading and preparing the data

In [9]:
# Loading dataset from the external data folder
data = pd.read_excel("../data/external/inflation_data.xlsx", sheet_name="data")

# Stripping leading and trailing whitespaces from column names
data.columns = [c.strip() for c in data.columns]

In [10]:
# Setting "Date" column type as datetime  
data["Date"] = pd.to_datetime(data["Date"])
data = data.sort_values("Date").reset_index(drop=True)

# Creating "Month" column for the later use for Random Forest model
data["Month"] = data["Date"].dt.month

# Creating dummy variables for month to be later used by linear benchmark models
month_dummies = pd.get_dummies(data['Month'], prefix='month', drop_first=True, dtype=int)

## Target selection and Test Set size

In [12]:
target_col = "Inflation"
window_size = 12
n_test_windows = 3
total_test_size = window_size * n_test_windows

# Defining functions

## ADF test

In [15]:
# Designing ADF test rules
def adf_test(series: pd.Series):
    s = series.dropna()
    if len(s) < 10:
        return np.nan
    try:
        _, pval, *_ = adfuller(s, autolag='AIC')
    except Exception:
        pval = np.nan
    return pval

In [16]:
# Adding interpretations to the ADF test
def interpret_adf(pval, alpha=0.05):
    if np.isnan(pval):
        return "ADF: p=NaN (test failed)"
    return "ADF: Reject unit root (stationary)" if pval < alpha else "ADF: Fail to reject unit root (non-stationary)"

In [17]:
# Defining function to generate stationarity report
def make_stationarity_report(df: pd.DataFrame, diff=False, alpha=0.05) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        if col == "Date" or not np.issubdtype(df[col].dtype, np.number):
            continue
        series = df[col].diff().dropna() if diff else df[col]
        label = "Diff(1)" if diff else "Level"
        adf_p = adf_test(series)
        rows.append({
            "Variable": col,
            "Series": label,
            "ADF_pvalue": adf_p,
            "ADF_interpretation": interpret_adf(adf_p, alpha)
        })
    return pd.DataFrame(rows)

## Reconstructing Inlfation from it's differenciated values

In [19]:
# Designing function to recreate true inflation levels from the differenciated values
def reconstruct_level_from_diff(last_level, diffs):
    out, curr = [], last_level
    for d in np.asarray(diffs).ravel():
        curr += d
        out.append(curr)
    return np.array(out)

In [20]:
# Designing function to recreate true inflation levels from twofold differenciated values
def reconstruct_level_from_diff2(last_level, last_diff, diff2s):
    diff1_preds = reconstruct_level_from_diff(last_diff, diff2s)
    level_preds = reconstruct_level_from_diff(last_level, diff1_preds)
    return level_preds

## MAPE function

In [22]:
# Creating MAPE measure
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denom = np.where(y_true == 0, 1e-6, np.abs(y_true))
    return np.mean(np.abs((y_true - y_pred) / denom))

# Exploratory Data Analysis

## Line plots

In [25]:
# Selecting numeric columns for further analysis
numeric_level_cols = [c for c in data.columns if c not in ["Date", "Month"] and np.issubdtype(data[c].dtype, np.number)]
levels_df = data[["Date"] + numeric_level_cols].copy()

# Iterating over the features to plot and save a line plot for each of them
for col in numeric_level_cols:
    plt.figure(figsize=(10, 5))
    plt.plot(levels_df["Date"], levels_df[col], linewidth=2, color='#333333')
    plt.title(f"{col} over time", fontsize=13)
    plt.xlabel("Date")
    plt.ylabel(col)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f"../reports/figures/eda_line_{col.replace(' ', '_')}.png", dpi=save_dpi, bbox_inches='tight')
    plt.close()

# Generating and saving a pairplot of economic indicators
try:
    pp = sns.pairplot(levels_df[numeric_level_cols].dropna(), corner=True, diag_kind="hist", 
                      plot_kws={'alpha': 0.6, 's': 20})
    pp.fig.suptitle("Pairplot of Economic Indicators", y=1.02, fontsize=14)
    plt.savefig("../reports/figures/eda_pairplot.png", dpi=save_dpi, bbox_inches='tight')
    plt.close()
except Exception:
    pass

## Correlation matrix

In [27]:
# Calculating and plotting the correlation matrix
corr_mat = levels_df[numeric_level_cols].corr(method='spearman', numeric_only=True).round(4)
corr_mat.to_csv("../reports/eda_corr_matrix.csv")

plt.figure(figsize=(10, 8))
sns.heatmap(corr_mat, annot=True, fmt=".4f", cmap="viridis", square=True, 
            cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix (Levels)", fontsize=14)
plt.tight_layout()
plt.savefig("../reports/figures/eda_corr_heatmap.png", dpi=save_dpi, bbox_inches='tight')
plt.close()

# Modelling

## Data preparation

In [30]:
# Preparing targeted datasets for linear and tree models
predictor_cols = ["Interest Rate", "M2", "Import", "Export", "Industry Production", "USDPLN", "USD Reserves"]

# Creating dataset for random forest model
data_tree = data.copy()

# Adding multiple lags and moving averages
for col in predictor_cols:
    if col == "Interest Rate":
        data_tree[f"{col}_lag6"] = data_tree[col].shift(6)
        data_tree[f"{col}_lag7"] = data_tree[col].shift(7)
        data_tree[f"{col}_lag8"] = data_tree[col].shift(8)
        data_tree[f"{col}_roll3"] = data_tree[col].shift(6).rolling(window=3).mean()
    else:
        data_tree[f"{col}_lag1"] = data_tree[col].shift(1)
        data_tree[f"{col}_lag2"] = data_tree[col].shift(2)
        data_tree[f"{col}_lag3"] = data_tree[col].shift(3)
        data_tree[f"{col}_roll3"] = data_tree[col].shift(1).rolling(window=3).mean()

# Adding autoregressive features for the target variable
data_tree[f"AutoReg_Lag1"] = data_tree[target_col].shift(1)
data_tree[f"AutoReg_Lag2"] = data_tree[target_col].shift(2)
data_tree[f"AutoReg_Lag3"] = data_tree[target_col].shift(3)

# Adding inflation momentum
data_tree[f"AutoReg_Diff1"] = data_tree[target_col].diff().shift(1)

# Removing raw predictors to prevent data leakage
data_tree = data_tree.drop(columns=predictor_cols)

# Creating dataset for linear and ARIMAX models
data_linear = data.copy()
data_linear = pd.concat([data_linear, month_dummies], axis=1).drop(columns=['Month'])

linear_diff_orders = {}

# Preparing variables for ADF test with target variable first
cols_to_test = [target_col] + predictor_cols
train_base_idx = len(data) - total_test_size

adf_report_data = []
for col in cols_to_test:
    train_series = data_linear[col].iloc[:train_base_idx].dropna()
    p_val_level = adf_test(train_series)
    
    # Determining stationarity status for the report
    if p_val_level < 0.05:
        stationarity_status = "Stationary"
        diff_order = 0
    else:
        diff1 = data_linear[col].diff()
        train_series_d1 = diff1.iloc[:train_base_idx].dropna()
        p_val_d1 = adf_test(train_series_d1)
        
        if p_val_d1 < 0.05:
            stationarity_status = "Order 1 differencing"
            diff_order = 1
        else:
            stationarity_status = "Order 2 differencing"
            diff_order = 2
            
    # Logging exact 3 columns to the report with 4 decimal places format
    adf_report_data.append({
        "Variable": col, 
        "P-value": f"{p_val_level:.4f}" if pd.notnull(p_val_level) else np.nan,
        "Stationarity": stationarity_status
    })
    
    # Transforming only predictor variables for the models
    # Keeping the dependent variable untouched
    if col in predictor_cols:
        linear_diff_orders[col] = diff_order
        if diff_order == 1:
            data_linear[col] = diff1
        elif diff_order == 2:
            data_linear[col] = diff1.diff()

# Saving ADF test report
adf_report_df = pd.DataFrame(adf_report_data)
adf_report_df.to_csv("../reports/adf_test_report.csv", index=False)

# Applying lags to transformed predictors
for col in predictor_cols:
    if col == "Interest Rate":
        data_linear[col] = data_linear[col].shift(6)
    else:
        data_linear[col] = data_linear[col].shift(1)

# Aligning dates across all datasets
valid_indices = data_tree.dropna().index.intersection(data_linear.dropna().index)

data_raw = data.loc[valid_indices].reset_index(drop=True)
data_tree = data_tree.loc[valid_indices].reset_index(drop=True)
data_linear = data_linear.loc[valid_indices].reset_index(drop=True)

# Calculating and saving VIF report
vif_test_df = data_linear[predictor_cols].dropna()
vif_report = pd.DataFrame()
vif_report["Variable"] = vif_test_df.columns
vif_report["VIF"] = [round(variance_inflation_factor(vif_test_df.values, i), 4) for i in range(vif_test_df.shape[1])]
vif_report.to_csv("../reports/vif_report.csv", index=False)

## Fitting the models

In [32]:
# Defining feature configurations for different models
base_vars = ["Interest Rate", "M2", "Import", "Export", "Industry Production"]
datasets_config = {
    "data_clean_1": base_vars,
    "data_clean_2": base_vars + ["USDPLN"],
    "data_clean_3": base_vars + ["USDPLN", "USD Reserves"]
}

In [33]:
detailed_results = [] 
rf_importance_data = {}
model_parameters_data = []

month_cols_linear = [c for c in data_linear.columns if c.startswith('month_')]

for name, vars_list in datasets_config.items():
    print(f"\nRunning {name}, Expanding-Window Validation ({n_test_windows} windows)")
    
    # Cleaning dataset name for the report (e.g. "dataset 1" -> "1")
    dataset_clean_name = str(name).replace("dataset ", "").replace("Dataset ", "").strip()
    
    linear_features = vars_list + month_cols_linear
    continuous_vars = [v for v in vars_list if v in data_linear.columns]
    
    # Standardizing variables directly before the model - scaling continuous variables only
    preprocessor = ColumnTransformer(
        transformers=[('num', StandardScaler(), continuous_vars)],
        remainder='passthrough'
    )
    
    # Building features dynamically for Random Forest
    tree_features = ["Month", "AutoReg_Lag1", "AutoReg_Lag2", "AutoReg_Lag3", "AutoReg_Diff1"]
    
    # Appending lags and rolling averages ONLY for variables in the current set
    for base_var in vars_list:
        tree_features.extend([c for c in data_tree.columns if c.startswith(base_var)])
        
    for w in range(n_test_windows):
        current_train_end = len(data_raw) - total_test_size + (w * window_size)
        current_test_end = current_train_end + window_size
        
        # Logging dates
        train_start = data_raw["Date"].iloc[0].strftime("%Y-%m")
        train_end = data_raw["Date"].iloc[current_train_end - 1].strftime("%Y-%m")
        test_start = data_raw["Date"].iloc[current_train_end].strftime("%Y-%m")
        test_end = data_raw["Date"].iloc[current_test_end - 1].strftime("%Y-%m")
        
        # Extracting the starting year to serve as the Forecast Window indicator
        forecast_year = data_raw["Date"].iloc[current_train_end].strftime("%Y")
        
        print(f"\n Window {w+1}/{n_test_windows} | Train: {train_start} to {train_end} | Forecast: {test_start} to {test_end}")

        # Retrieving true targets (levels) for evaluation
        y_true_level = data_raw.iloc[current_train_end:current_test_end][target_col].values
        
        # Tuning Random Forest
        x_train_tree = data_tree.iloc[:current_train_end][tree_features]
        y_train_tree = data_tree.iloc[:current_train_end][target_col]
        
        tscv = TimeSeriesSplit(n_splits=3)
        rf_search = BayesSearchCV(
            estimator=RandomForestRegressor(random_state=42),
            search_spaces={
                'n_estimators': Integer(100, 600),
                'max_depth': Integer(2, 10), 
                'min_samples_split': Integer(2, 15),
                'min_samples_leaf': Integer(2, 10),
                'max_features': Categorical(['sqrt', 'log2'])
            },
            n_iter=15, 
            cv=tscv, scoring='neg_root_mean_squared_error',
            n_jobs=-1, verbose=0, random_state=42
        )
        rf_search.fit(x_train_tree, y_train_tree)
        best_rf_params = rf_search.best_params_
        print(f" [Random Forest] Best Params: {best_rf_params}")

        # Executing direct forecasting
        lr_pred_stat = []
        rf_pred_stat = []
        
        # Variables to store Linear Regression parameters for the report
        current_r_squared = np.nan
        current_intercept = np.nan
        
        for h in range(1, window_size + 1):
            
            # Linear Regression
            x_train_h_lin_raw = data_linear.iloc[:current_train_end].shift(h - 1)[linear_features]
            y_train_h_lin_raw = data_raw[target_col].iloc[:current_train_end]
            
            valid_idx_lin = x_train_h_lin_raw.dropna().index.intersection(y_train_h_lin_raw.dropna().index)
            
            x_train_h_lin = x_train_h_lin_raw.loc[valid_idx_lin]
            y_train_h_lin = y_train_h_lin_raw.loc[valid_idx_lin]
            x_test_h_lin = data_linear.iloc[[current_train_end]][linear_features]
            
            # Fitting standard OLS with preprocessor
            lr_h = Pipeline([("preprocessor", preprocessor), ("lr", LinearRegression())])
            lr_h.fit(x_train_h_lin, y_train_h_lin)
            
            level_pred = lr_h.predict(x_test_h_lin)[0]
            lr_pred_stat.append(level_pred)
            
            if h == 1:
                r_squared = lr_h.score(x_train_h_lin, y_train_h_lin)
                intercept = lr_h.named_steps['lr'].intercept_
                current_r_squared = float(r_squared)
                current_intercept = float(intercept)
                
                passthrough_vars = [c for c in x_train_h_lin.columns if c not in continuous_vars]
                all_features = continuous_vars + passthrough_vars
                coefs = pd.Series(lr_h.named_steps['lr'].coef_, index=all_features)
                top_coefs = coefs.abs().sort_values(ascending=False).head(2).index.tolist()
                print(f" [Linear Regression] R^2={r_squared:.4f}, Intercept={intercept:.4f}, Top Drivers: {', '.join(top_coefs)}")

            # Random Forest Model
            x_train_h_tree_raw = data_tree.iloc[:current_train_end].shift(h - 1)[tree_features]
            y_train_tree_raw = data_tree.iloc[:current_train_end][target_col]
            
            valid_idx_tree = x_train_h_tree_raw.dropna().index.intersection(y_train_tree_raw.dropna().index)
            
            x_train_h_tree = x_train_h_tree_raw.loc[valid_idx_tree]
            y_train_h_tree = y_train_tree_raw.loc[valid_idx_tree]
            x_test_h_tree = data_tree.iloc[[current_train_end]][tree_features]
            
            rf_h = RandomForestRegressor(random_state=42, **best_rf_params)
            rf_h.fit(x_train_h_tree, y_train_h_tree)
            rf_pred_stat.append(rf_h.predict(x_test_h_tree)[0])
            
            if h == 1 and w == n_test_windows - 1:
                imp_df = pd.DataFrame({
                    "Feature": x_train_h_tree.columns,
                    "Importance": rf_h.feature_importances_
                }).sort_values("Importance", ascending=False).head(10)
                rf_importance_data[name] = imp_df

        # ARIMAX Model
        try:
            y_train_arimax = data_raw.iloc[:current_train_end][target_col]
            x_train_arimax = data_linear.iloc[:current_train_end][linear_features]
            
            valid_arimax_idx = x_train_arimax.dropna().index
            y_train_arimax = y_train_arimax.loc[valid_arimax_idx]
            x_train_arimax = x_train_arimax.loc[valid_arimax_idx]
            
            arimax = pm.auto_arima(y_train_arimax.values, X=x_train_arimax.values, 
                                   test='adf', seasonal=False, 
                                   stepwise=True, suppress_warnings=True)
            
            x_test_arimax = np.tile(data_linear.iloc[current_train_end][linear_features].values, (window_size, 1))
            arimax_pred_stat = arimax.predict(n_periods=window_size, X=x_test_arimax)
            arimax_order = arimax.order
            print(f" [ARIMAX] Best Order (p,d,q): {arimax_order}")
        except Exception:
            arimax_pred_stat = np.full(window_size, np.nan)
            arimax_order = "(N/A)"
            print("  >> [ARIMAX] Failed to fit.")

        lr_pred_stat = np.array(lr_pred_stat)
        rf_pred_stat = np.array(rf_pred_stat)

        # Keys explicitly matched to the final report structure
        models_preds_level = {
            "Linear Regression": lr_pred_stat,
            "Auto ARIMAX": arimax_pred_stat, 
            "Random Forest": rf_pred_stat
        }

        # Calculating evaluation metrics and appending to detailed results
        for model_name, y_pred in models_preds_level.items():
            rmse = float(np.sqrt(mean_squared_error(y_true_level, y_pred)))
            mae = float(mean_absolute_error(y_true_level, y_pred))
            mape_score = float(mape(y_true_level, y_pred))
            
            detailed_results.append({
                "Dataset": dataset_clean_name,
                "Forecast Window": forecast_year,
                "Model": model_name,
                "MAE": round(mae, 4),
                "RMSE": round(rmse, 4),
                "MAPE": round(mape_score, 4)
            })
            
        # Appending data for the model parameters report
        model_parameters_data.append({
            "Dataset": dataset_clean_name,
            "Forecast Window": forecast_year,
            "R^2": f"{current_r_squared:.4f}",
            "Intercept": f"{current_intercept:.4f}",
            "(p,d,q)": str(arimax_order),
            "n_estimators": best_rf_params.get('n_estimators'),
            "max_depth": best_rf_params.get('max_depth'),
            "min_samples_split": best_rf_params.get('min_samples_split'),
            "min_samples_leaf": best_rf_params.get('min_samples_leaf'),
            "max_features": best_rf_params.get('max_features')
        })

# Generating and saving model parameters report with MultiIndex structure
model_params_df = pd.DataFrame(model_parameters_data)
model_params_df.to_csv("../reports/model_parameters_report.csv", index=False)

# Generating and saving detailed error table (results for each window separately)
errors_detailed_df = pd.DataFrame(detailed_results)
errors_detailed_df.to_csv("../reports/errors_detailed_all.csv", index=False)


Running data_clean_1, Expanding-Window Validation (3 windows)

 Window 1/3 | Train: 2006-09 to 2022-12 | Forecast: 2023-01 to 2023-12
 [Random Forest] Best Params: OrderedDict({'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 4, 'n_estimators': 600})
 [Linear Regression] R^2=0.4009, Intercept=2.9071, Top Drivers: Interest Rate, month_9
 [ARIMAX] Best Order (p,d,q): (1, 1, 1)

 Window 2/3 | Train: 2006-09 to 2023-12 | Forecast: 2024-01 to 2024-12
 [Random Forest] Best Params: OrderedDict({'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100})
 [Linear Regression] R^2=0.3360, Intercept=3.3727, Top Drivers: Interest Rate, month_9
 [ARIMAX] Best Order (p,d,q): (2, 1, 3)

 Window 3/3 | Train: 2006-09 to 2024-12 | Forecast: 2025-01 to 2025-12
 [Random Forest] Best Params: OrderedDict({'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100})
 [Linear 

## Feature importance

In [35]:
# Plotting combined feature importance
if len(rf_importance_data) > 0:
    n_plots = len(rf_importance_data)
    fig, axes = plt.subplots(1, n_plots, figsize=(15, 6), sharey=False) 
    
    # Ensuring axes is iterable even if there is only one dataset
    if n_plots == 1:
        axes = [axes]
        
    plot_titles = ["Base Features", "Base + USDPLN", "Base + USDPLN + USD Reserves"]

    # Iterating directly over the keys stored in the dictionary during training
    for ax, key, title in zip(axes, list(rf_importance_data.keys()), plot_titles):
        df_imp = rf_importance_data[key]
        
        sns.barplot(data=df_imp, x="Importance", y="Feature", ax=ax, 
                    color=rf_plot_color, orient='h', alpha=0.8)
        
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel("Importance Score", fontsize=10)
        # Removing y-label to save space
        ax.set_ylabel("") 
        
    plt.tight_layout()
    plt.savefig("../reports/figures/rf_feature_importance_comparison.png", dpi=save_dpi, bbox_inches='tight')
    plt.close()